## Similarity Maps of FDG-PET data

In [ ]:
#Import statements
import torch
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
from sklearn.neighbors import NearestNeighbors
import nilearn
from nilearn import plotting, image, surface, datasets
import copy
import scipy.sparse as sp
from scipy.sparse import csr_matrix
from matplotlib.colors import LinearSegmentedColormap
import ants
print(torch.__version__)

#### Input non-labeled training, testing datasets, and diagnosis-labeled test data

In [ ]:
train_csv = pd.read_csv('model_data/train_brain_labels_deid.csv', index_col=0)
train_parquet = pd.read_parquet('model_data/train_brain_data_deid.parquet')
test_csv = pd.read_csv('model_data/test_brain_labels_deid.csv', index_col=0)
test_parquet = pd.read_parquet('model_data/test_brain_data_deid.parquet')

nd_csv = pd.read_csv('model_data/nd_brain_labels_deid.csv', index_col=0)
nd_parquet = pd.read_parquet('model_data/nd_brain_data_deid.parquet')

#### Retain only data with diagnosis of ad, bvftd, cu, or dlb, and age and sex information

In [ ]:
desired_columns = ['sex_female', 'age_at_scan', 'ad', 'bvftd', 'cu', 'dlb']
nd_csv_short = nd_csv[desired_columns].copy()
removed_columns = list(set(nd_csv.columns) - set(desired_columns))
for col in removed_columns:
    if col in nd_csv:
        mask = (nd_csv[col] != 1)
        mask = mask.loc[nd_csv_short.index]
        nd_csv_short = nd_csv_short[mask]
nd_csv_short.to_csv('model_data/nd_filtered_data.csv', index=False)

#### Load materials

In [ ]:
brain_mask = nilearn.image.load_img('model_data/brain_mask.nii')
bg = nilearn.image.load_img('model_data/t1.nii')
mcalt = nilearn.image.load_img('model_data/mcalt_t1.nii')
mni = nilearn.image.load_img('model_data/mni_t1.nii')
ants_mcalt = ants.from_nibabel(mcalt)
ants_mni = ants.from_nibabel(mni)
reg_dict = ants.registration(fixed=ants_mni, moving=ants_mcalt, type_of_transform='SyN')
ants_transform = reg_dict['fwdtransforms']

#### Visualize 2D slices with background image

In [ ]:
img_data = np.zeros(brain_mask.shape)
vec = nd_parquet.iloc[0, 0]
nz_indices = np.ma.nonzero(brain_mask.get_fdata())
img_data[nz_indices] = vec

In [ ]:
img_new = nilearn.image.new_img_like(brain_mask, img_data)
nilearn.plotting.plot_img(img_new, cut_coords=(0, 0, 0), bg_img=mcalt, alpha=0.5)

In [ ]:
ants_img_new = ants.from_nibabel(img_new)
ants_trans_img_new = ants.apply_transforms(fixed=ants_mni, moving=ants_img_new, transformlist=ants_transform)
trans_img_new = ants.to_nibabel(ants_trans_img_new)
nilearn.plotting.plot_img(trans_img_new, cut_coords=(0, 0, 0), bg_img=mni, alpha=0.5)

#### Surface rendering

In [ ]:
def get_surf_plot(data, hemi='right', view='lateral', mask=brain_mask, interactive=False, threshold=0.05, color_map='cold_hot', colorbar=True, transform=ants_transform, fixed=ants_mni, moving=ants_mcalt):
    img_data = np.zeros(mask.shape)
    nz_indices = np.ma.nonzero(brain_mask.get_fdata())
    img_data[nz_indices] = data
    
    mean_value = np.mean(img_data[nz_indices])
    img_data[nz_indices] -= mean_value
    img = nilearn.image.new_img_like(brain_mask, img_data)    
    img_smooth = nilearn.image.smooth_img(img, fwhm=6)
    ants_img = ants.from_nibabel(img_smooth)
    ants_trans_img = ants.apply_transforms(fixed=ants_mni, moving=ants_img, transformlist=ants_transform)
    trans_img = ants.to_nibabel(ants_trans_img)
    fsaverage = datasets.fetch_surf_fsaverage()
    mesh = surface.load_surf_mesh(fsaverage.pial_right)
    bg_map = fsaverage.sulc_right
    texture = surface.vol_to_surf(trans_img, mesh)

    if interactive == False:
        fig = plotting.plot_surf_stat_map(mesh, texture, hemi=hemi, view=view, colorbar=colorbar, threshold=threshold, bg_map=fsaverage.sulc_right,cmap=color_map)                       
    else:
        fig = plotting.plot_surf_stat_map(mesh, texture, hemi=hemi, view=view, colorbar=colorbar, threshold=threshold, bg_map=fsaverage.sulc_right,cmap=color_map, engine='plotly')

    return fig

#### Surface renderings 

In [ ]:
figure = get_surf_plot(vec, view='lateral', color_map='turbo', threshold=None)

In [ ]:
test_reconstructed = pd.read_parquet('model_data/recon_test_data.parquet')
figure = get_surf_plot(test_reconstructed.iloc[0].values, view='lateral', color_map='turbo', threshold=None)

#### Similarity map

In [ ]:
def get_sim_plot(data, hemi='right', view='lateral', mask=brain_mask, interactive=False, threshold=0.05, color_map='cold_hot', colorbar=True, transform=ants_transform, fixed=ants_mni, moving=ants_mcalt):
    img_data = np.zeros(mask.shape)
    nz_indices = np.ma.nonzero(brain_mask.get_fdata())
    img_data[nz_indices] = data
    
    img = nilearn.image.new_img_like(brain_mask, img_data)
    img_smooth = nilearn.image.smooth_img(img, fwhm=6)
    ants_img = ants.from_nibabel(img_smooth)
    ants_trans_img = ants.apply_transforms(fixed=ants_mni, moving=ants_img, transformlist=ants_transform)
    trans_img = ants.to_nibabel(ants_trans_img)
    fsaverage = datasets.fetch_surf_fsaverage()
    mesh = surface.load_surf_mesh(fsaverage.pial_right)
    bg_map = fsaverage.sulc_right
    texture = surface.vol_to_surf(trans_img, mesh)
    
    if interactive == False:
        fig = plotting.plot_surf_stat_map(mesh, texture, hemi=hemi, view=view, colorbar=colorbar, threshold=threshold, bg_map=fsaverage.sulc_right,cmap=color_map)                       
    else:
        fig = plotting.plot_surf_stat_map(mesh, texture, hemi=hemi, view=view, colorbar=colorbar, threshold=threshold, bg_map=fsaverage.sulc_right,cmap=color_map, engine='plotly')

    return fig

In [ ]:
def voxel_intensity_difference(array1, array2):
    array1 = np.array(array1)
    array2 = np.array(array2)   
    if array1.shape != array2.shape:
        raise ValueError("Both arrays must have the same length")
    difference = np.abs(array1 - array2)
    
    return difference

In [ ]:
img_1 = nd_parquet.iloc[0, 0]
img_2 = nd_parquet.iloc[846, 0]

In [ ]:
img_overlap = voxel_intensity_difference(img_1, img_2)

In [ ]:
fig = get_sim_plot(img_overlap, view="lateral", color_map='RdGy', threshold=None)

#### Surface rendering from file path

In [ ]:
def get_path_plot(path, hemi='right', view='lateral', mask=brain_mask, interactive=False, threshold=0.05, color_map='cold_hot', colorbar=True, transform=ants_transform, fixed=ants_mni, moving=ants_mcalt):
    img = nilearn.image.load_img(path)
    img_smooth = nilearn.image.smooth_img(img, fwhm=6)
    ants_img = ants.from_nibabel(img_smooth)
    ants_trans_img = ants.apply_transforms(fixed=ants_mni, moving=ants_img, transformlist=ants_transform)
    trans_img = ants.to_nibabel(ants_trans_img)
    fsaverage = datasets.fetch_surf_fsaverage()
    mesh = surface.load_surf_mesh(fsaverage.pial_right)
    bg_map = fsaverage.sulc_right
    texture = surface.vol_to_surf(trans_img, mesh)

    if interactive == False:
        fig = plotting.plot_surf_stat_map(mesh, texture, hemi=hemi, view=view, colorbar=colorbar, threshold=threshold, bg_map=fsaverage.sulc_right,cmap=color_map)                       
    else:
        fig = plotting.plot_surf_stat_map(mesh, texture, hemi=hemi, view=view, colorbar=colorbar, threshold=threshold, bg_map=fsaverage.sulc_right,cmap=color_map, engine='plotly')

    return fig

In [ ]:
figure = get_path_plot(path='path_files/dim_5_i_-2.nii.gz', view='medial', threshold=None, color_map='turbo')
figure.show()